# symb-guided-dis inference

This notebook runs reconstruction, one-shot timbre transfer, and timbre interpolation from examples in a prepared LMDB dataset.

It intentionally uses only repository-relative code plus two user-supplied paths: a prepared dataset and a run directory produced by `scripts/train.py`. It contains no local-audio or legacy-configuration workflow.


In [ ]:
# Setup: set paths through environment variables or directly below.
import os
import sys
from pathlib import Path

import gin
import numpy as np
import torch
try:
    from IPython.display import Audio, display
except ImportError:
    Audio = None

    def display(value):
        return value


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src' / 'symb_guided_dis').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from the symb-guided-dis repository or one of its subdirectories.')


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import symb_guided_dis.training.model as model_module
from symb_guided_dis.dataset.dataset import BaseDataset


# Either export these variables before opening Jupyter or set the strings here.
DATASET_PATH = os.environ.get('SYMB_GUIDED_DIS_DATASET_PATH', '')
RUN_DIR = os.environ.get('SYMB_GUIDED_DIS_RUN_DIR', '')
CHECKPOINT_NAME = 'best_model.pt'
EXTRA_ARRAY_KEYS = ()  # For example: ('clap', 'clap_timbre_aug') for the CLAP configuration.

# Positions within the validation split. They fall back to all examples if no validation split exists.
SAMPLE_A_POSITION = 0
SAMPLE_B_POSITION = 1

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PLAYBACK_SR = 44_100
RECONSTRUCTION_STEPS = 200
TRANSFER_STEPS = 200
INTERPOLATION_STEPS = 200
RECONSTRUCTION_SEED = 0
TRANSFER_SEED = 6
INTERPOLATION_SEED = 0

print(f'Repository: {REPO_ROOT}')
print(f'Device: {DEVICE}')
print('Set SYMB_GUIDED_DIS_DATASET_PATH and SYMB_GUIDED_DIS_RUN_DIR before running the next cells.')


## Load a current run and prepared LMDB

The run directory must contain `config.gin` and the selected checkpoint. The configuration is parsed directly, so this notebook is intentionally for runs created with the current `symb_guided_dis` package.


In [ ]:
# Load model and checkpoint.
if not RUN_DIR:
    raise ValueError('Set SYMB_GUIDED_DIS_RUN_DIR or RUN_DIR to a run directory created by scripts/train.py.')

run_dir = Path(RUN_DIR).expanduser().resolve()
config_path = run_dir / 'config.gin'
checkpoint_path = run_dir / CHECKPOINT_NAME
if not config_path.is_file():
    raise FileNotFoundError(f'Run configuration not found: {config_path}')
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

gin.enter_interactive_mode()
gin.clear_config()
gin.parse_config_file(config_path)
model_class = gin.query_parameter('%model_class').configurable.wrapped
if model_class is not model_module.RectifiedFlow:
    raise ValueError(f'Expected a RectifiedFlow run, got {model_class.__name__}.')

model = model_class().to(DEVICE)
checkpoint = torch.load(checkpoint_path, map_location='cpu')
state_dict = checkpoint.get('ema_state_dict') or checkpoint.get('model_state_dict')
if state_dict is None:
    raise KeyError('Checkpoint has neither ema_state_dict nor model_state_dict.')
model._load_state_dict_compat(state_dict, strict=True)
model.eval()
codec = model.build_latent_codec(device=str(DEVICE))

if getattr(model, 'latent_normalization', None) == 'dataset_per_channel' and not model.has_fixed_latent_stats():
    raise RuntimeError('This checkpoint has no stored dataset-wide latent normalization statistics.')

print(f'Loaded {checkpoint_path.name} from {run_dir}')
print(f'timbre_input_key={model.timbre_input_key} | solver={model.flow_solver}')


In [ ]:
# Load the prepared LMDB and select two examples.
if not DATASET_PATH:
    raise ValueError('Set SYMB_GUIDED_DIS_DATASET_PATH or DATASET_PATH to a prepared LMDB directory.')

dataset_path = Path(DATASET_PATH).expanduser().resolve()
dataset = BaseDataset(
    str(dataset_path),
    return_latents=True,
    return_midi=True,
    return_struct_aug_latent=False,
    return_timbre_aug_latent=False,
    extra_array_keys=EXTRA_ARRAY_KEYS,
)
metadata = dataset.get_all_metadata()
validation_indices = [index for index, item in enumerate(metadata) if item.get('split') == 'validation']
candidate_indices = validation_indices or list(range(len(dataset)))
if not candidate_indices:
    raise ValueError(f'No examples found in {dataset_path}.')

index_a = candidate_indices[SAMPLE_A_POSITION % len(candidate_indices)]
index_b = candidate_indices[SAMPLE_B_POSITION % len(candidate_indices)]
sample_a, sample_b = dataset[index_a], dataset[index_b]

def describe(name, index, sample):
    item = sample['metadata']
    print(f"{name}: index={index}, split={item.get('split')}, track={item.get('track')}, stem={item.get('stem')}")

describe('A', index_a, sample_a)
describe('B', index_b, sample_b)


## Reconstruction and transfer

Reconstruction uses each example for both structure and timbre. Transfer combines the structure of one example with the timbre of the other.


In [ ]:
def to_batch_latent(value):
    latent = torch.as_tensor(value, device=DEVICE, dtype=torch.float32)
    if latent.ndim == 2:
        latent = latent.unsqueeze(0)
    if latent.ndim != 3 or latent.shape[1] != model.latent_dim:
        raise ValueError(f'Expected latent shape [B, {model.latent_dim}, T], got {tuple(latent.shape)}.')
    return latent


def play_audio(label, audio):
    waveform = torch.as_tensor(audio).detach().cpu().squeeze().numpy().astype(np.float32)
    print(label)
    if Audio is None:
        print(f'Audio playback requires IPython; waveform shape={waveform.shape}')
    else:
        display(Audio(waveform, rate=PLAYBACK_SR))


def decode(latent):
    return model.decode_model_latent_to_audio(latent, codec=codec)


def timbre_source(sample):
    key = str(model.timbre_input_key)
    if key == 'none':
        return None
    latent = to_batch_latent(sample['latent'])
    latent_model = model.normalize_latent_for_model(latent)
    if key == 'latent':
        return latent_model
    if key == 'latent_mean':
        return latent_model.mean(dim=-1)
    if key == 'latent_mean_std':
        return torch.cat([latent_model.mean(dim=-1), latent_model.std(dim=-1)], dim=1)
    if key not in sample:
        raise KeyError(f"The LMDB sample has no '{key}' array. Add it to EXTRA_ARRAY_KEYS.")
    value = torch.as_tensor(sample[key], device=DEVICE, dtype=torch.float32)
    return value.unsqueeze(0) if value.ndim == 1 else value


@torch.no_grad()
def generate(structure_latent, target_timbre_source, *, steps, seed):
    structure_model, latent_stats = model.normalize_latent_for_model(
        structure_latent, return_stats=True
    )
    structure_embedding = model.encode_structure(structure_model)
    timbre_embedding = (
        None if target_timbre_source is None else model.encode_timbre(target_timbre_source)
    )
    return model.sample_with_embeddings(
        structure_embedding=structure_embedding,
        timbre_embedding=timbre_embedding,
        latent_shape=tuple(structure_model.shape),
        num_inference_steps=int(steps),
        solver=model.flow_solver,
        guidance_timbre=model.flow_guidance_timbre,
        guidance_structure=model.flow_guidance_structure,
        seed=int(seed),
        output_space='raw',
        latent_stats=latent_stats,
    )


latent_a, latent_b = to_batch_latent(sample_a['latent']), to_batch_latent(sample_b['latent'])
timbre_a, timbre_b = timbre_source(sample_a), timbre_source(sample_b)

reconstruction_a = generate(latent_a, timbre_a, steps=RECONSTRUCTION_STEPS, seed=RECONSTRUCTION_SEED)
reconstruction_b = generate(latent_b, timbre_b, steps=RECONSTRUCTION_STEPS, seed=RECONSTRUCTION_SEED)
transfer_ab = generate(latent_a, timbre_b, steps=TRANSFER_STEPS, seed=TRANSFER_SEED)
transfer_ba = generate(latent_b, timbre_a, steps=TRANSFER_STEPS, seed=TRANSFER_SEED)

play_audio('A reference', decode(latent_a))
play_audio('A reconstruction', decode(reconstruction_a))
play_audio('B reference', decode(latent_b))
play_audio('B reconstruction', decode(reconstruction_b))
play_audio('A structure + B timbre', decode(transfer_ab))
play_audio('B structure + A timbre', decode(transfer_ba))


## Timbre interpolation

This fixes A's structure and linearly interpolates between A and B in the model's timbre-embedding space.


In [ ]:
@torch.no_grad()
def generate_from_timbre_embedding(structure_latent, timbre_embedding, *, steps, seed):
    structure_model, latent_stats = model.normalize_latent_for_model(
        structure_latent, return_stats=True
    )
    return model.sample_with_embeddings(
        structure_embedding=model.encode_structure(structure_model),
        timbre_embedding=timbre_embedding,
        latent_shape=tuple(structure_model.shape),
        num_inference_steps=int(steps),
        solver=model.flow_solver,
        guidance_timbre=model.flow_guidance_timbre,
        guidance_structure=model.flow_guidance_structure,
        seed=int(seed),
        output_space='raw',
        latent_stats=latent_stats,
    )


if timbre_a is None or timbre_b is None:
    print('This configuration has no timbre conditioning; interpolation is unavailable.')
else:
    embedding_a = model.encode_timbre(timbre_a)
    embedding_b = model.encode_timbre(timbre_b)
    for alpha in torch.linspace(0.0, 1.0, steps=5).tolist():
        embedding = torch.lerp(embedding_a, embedding_b, alpha)
        generated = generate_from_timbre_embedding(
            latent_a,
            embedding,
            steps=INTERPOLATION_STEPS,
            seed=INTERPOLATION_SEED,
        )
        play_audio(f'A structure + interpolated timbre (alpha={alpha:.2f})', decode(generated))
